# TAIX-Ray: Temporal Patient-Aware → Causal Decoder

Longitudinal chest X-ray severity prediction from parquet files.

**Key features:**
- Streaming parquet loading — never loads >1 shard at once; uses lazy row-indexed reads
- TorchXRayVision DenseNet-121 backbone (pretrained on 5 CXR datasets)
- Time2Vec + TemporalTransformer (bidirectional/causal with [ABSENT] tokens)
- Causal GAT with fixed clinical DAG + do-calculus (N2)
- Physician-aware CORAL threshold bias + deep ensemble (N3)
- Autoregressive progression forecasting (N4)
- tqdm progress bars for all loading phases

## 1. Imports

In [1]:
!pip install lightning torchxrayvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 58.2 MB/s eta 0:00:00


In [2]:
import os, sys, math, json, copy, warnings, random, io
from pathlib import Path
from typing import Optional, Callable, Any
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

import lightning as L
from lightning.pytorch import LightningModule, LightningDataModule
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

from PIL import Image
import scipy.stats
from sklearn.metrics import roc_auc_score

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

try:
    import torchxrayvision as xrv
    HAS_XRV = True
except ImportError:
    HAS_XRV = False
    print('WARNING: torchxrayvision not installed. Run: pip install torchxrayvision')

warnings.filterwarnings('ignore')
torch.set_float32_matmul_precision('medium')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, Lightning {L.__version__}, Device: {device}')

PyTorch 2.10.0+cu128, Lightning 2.6.5, Device: cuda


## 2. Configuration

Adjust column names to match your parquet schema.

In [3]:
class Config:
    parquet_dir = Path('/kaggle/input/datasets/ahmedalkasaby/taix-ray-dataset')
    parquet_pattern = '*.parquet'
    img_size = 224
    max_seq_len = 8

    col_patient_id = 'PatientID'
    col_study_date = 'StudyDate'
    col_physician_id = 'PhysicianID'
    col_image = 'image'

    findings = [
        'HeartSize', 'PulmonaryCongestion',
        'PleuralEffusion_Right', 'PleuralEffusion_Left',
        'PulmonaryOpacities_Right', 'PulmonaryOpacities_Left',
        'Atelectasis_Right', 'Atelectasis_Left',
    ]
    ordinal_levels = 5

    backbone_name = 'densenet121-res224-all'
    feat_dim = 1024

    d_model = 256
    nhead = 4
    transformer_layers = 2
    dim_feedforward = 512
    dropout = 0.1
    gat_heads = 4
    gat_dim = 64
    num_thresholds = 4

    sup_epochs = 3
    sup_batch_size = 4
    sup_lr = 1e-4
    sup_weight_decay = 1e-5
    forecast_weight = 0.0

    num_physicians = 500
    physician_emb_dim = 32

    sigmas = [60, 90, 180, 365, 730]
    default_sigma = 180
    ensemble_seeds = [42]
    moco_queue_size = 4096
    moco_temperature = 0.07
    moco_momentum = 0.999
    output_dir = Path('/kaggle/working/')
    num_folds = 5

cfg = Config()

## A. Parquet Schema Check

In [4]:
files = sorted(cfg.parquet_dir.glob(cfg.parquet_pattern))
print("Parquet dir:", cfg.parquet_dir)
print("Pattern:", cfg.parquet_pattern)
print("Found", len(files), "parquet files")

first = files[0]
schema = pq.read_schema(first)

print("=== Parquet Schema ===")
for f in schema:
    print(f"  {f.name}: {f.type}")

columns = [f.name for f in schema]

if "Image" in columns:
    cfg.col_image = "Image"
elif "image" in columns:
    cfg.col_image = "image"
else:
    raise ValueError("No image column found. Expected Image or image.")

print(f'\nUsing image column: "{cfg.col_image}"')
print(f'Image column "{cfg.col_image}" exists:', cfg.col_image in columns)

# Read only first row safely using pyarrow
table = pq.read_table(first, columns=[cfg.col_image])
v = table[cfg.col_image][0].as_py()

print("Image cell type:", type(v).__name__)

if isinstance(v, dict):
    print("Image dict keys:", v.keys())
elif hasattr(v, "keys"):
    print("Image keys:", list(v.keys()))
else:
    print("Image preview:", str(v)[:200])

Parquet dir: /kaggle/input/datasets/ahmedalkasaby/taix-ray-dataset
Pattern: *.parquet
Found 59 parquet files
=== Parquet Schema ===
  UID: string
  Fold: int64
  Split: string
  PatientID: string
  PhysicianID: string
  StudyDate: string
  Age: int64
  Sex: string
  HeartSize: int64
  PulmonaryCongestion: int64
  PleuralEffusion_Right: int64
  PleuralEffusion_Left: int64
  PulmonaryOpacities_Right: int64
  PulmonaryOpacities_Left: int64
  Atelectasis_Right: int64
  Atelectasis_Left: int64
  Image: struct<bytes: binary, path: string>

Using image column: "Image"
Image column "Image" exists: True
Image cell type: dict
Image dict keys: dict_keys(['bytes', 'path'])


## 3. Streaming Parquet Dataset (Memory-Safe)

**Loading strategy (3 phases):**
1. **Scan** — read metadata columns only (PatientID, StudyDate, findings) from ALL shards → build patient index. ~100MB for 100K rows.
2. **Sequence assembly** — group visits by PatientID, sort by date. No images involved.
3. **Lazy image loading** — `__getitem__` reads specific rows from parquet on-demand using PyArrow's indexed reads. Each shard is cached after first access (LRU, max 2 shards).

Peak memory: ~1 shard worth of image data (~2-4GB) + decoded tensors in the DataLoader batch.

In [5]:
class ParquetCXRDataset(Dataset):
    '''Memory-safe longitudinal CXR dataset from parquet.
    Loads metadata only during init; decodes images lazily in __getitem__.'''
    def __init__(self, parquet_dir, parquet_pattern='*.parquet',
                 findings=None, max_seq_len=64, img_size=224,
                 transform=None, fold=None, fraction=1.0, seed=42):
        self.findings = findings or []
        self.max_seq_len = max_seq_len
        self.img_size = img_size
        self.transform = transform or self._default_transform()
        self.shard_paths = sorted(Path(parquet_dir).glob(parquet_pattern))
        if not self.shard_paths:
            raise FileNotFoundError(f'No parquet files in {parquet_dir}/{parquet_pattern}')

        # Phase 1: Scan metadata only (NO image column) from all shards
        # This stays small: ~100MB for 100K rows of PatientID + dates + labels
        meta_cols = [cfg.col_patient_id, cfg.col_study_date,
                     cfg.col_physician_id] + self.findings
        # Filter to columns that actually exist
        available_cols = self._check_columns(meta_cols)

        pid_to_visits = defaultdict(list)
        for shard_idx, spath in enumerate(tqdm(self.shard_paths, desc='Scanning metadata')):
            # Read only the metadata columns (tiny)
            df = pd.read_parquet(spath, columns=available_cols)
            for row_idx, (_, row) in enumerate(df.iterrows()):
                pid = row[cfg.col_patient_id]
                date = row[cfg.col_study_date]
                if isinstance(date, str):
                    date = datetime.strptime(date, '%Y-%m-%d')
                physician_id_raw = row.get(cfg.col_physician_id, -1)
                try:
                    physician_id = int(physician_id_raw)
                except (ValueError, TypeError):
                    physician_id = abs(hash(str(physician_id_raw))) % cfg.num_physicians
                labels = [int(row.get(f, -1)) for f in self.findings]
                pid_to_visits[pid].append({
                    'date': date,
                    'physician_id': physician_id,
                    'labels': labels,
                    'shard_idx': shard_idx,
                    'row_idx': row_idx,
                })
            del df  # free per-shard dataframe immediately

        # Apply fold filtering on the index
        if fold is not None:
            pids = list(pid_to_visits.keys())
            pids = [p for p in pids if hash(str(p)) % cfg.num_folds == fold]
            pid_to_visits = {p: pid_to_visits[p] for p in pids}

        # Fraction sampling at patient level
        if fraction < 1.0:
            rng = random.Random(seed)
            pids = list(pid_to_visits.keys())
            rng.shuffle(pids)
            keep = set(pids[:max(1, int(len(pids) * fraction))])
            pid_to_visits = {p: pid_to_visits[p] for p in pids if p in keep}

        # Phase 2: Build patient sequences (still no images)
        # Drop patients with < 2 visits — no meaningful longitudinal signal
        self.patient_groups = []
        for pid, visits in tqdm(pid_to_visits.items(), desc='Assembling sequences'):
            if len(visits) < 2:
                continue
            visits.sort(key=lambda v: v['date'])
            self.patient_groups.append({
                'patient_id': pid,
                'length': len(visits),
                'dates': [v['date'] for v in visits],
                'labels': [v['labels'] for v in visits],
                'physician_ids': [v['physician_id'] for v in visits],
                'shard_idxs': [v['shard_idx'] for v in visits],
                'row_idxs': [v['row_idx'] for v in visits],
            })

        # Instance-level LRU cache (not shared across instances)
        self._shard_cache = {}
        self._shard_cache_order = []

        # Free the big dict
        del pid_to_visits

        lengths = [g['length'] for g in self.patient_groups]
        print(f'Patients: {len(self.patient_groups)}, '
              f'Visits: min={min(lengths)} mean={np.mean(lengths):.1f} '
              f'median={np.median(lengths):.0f} max={max(lengths)}')

    _schema_cache = None

    @classmethod
    def _check_columns(cls, required_cols):
        if cls._schema_cache is None:
            first = sorted(Path(cfg.parquet_dir).glob(cfg.parquet_pattern))[0]
            cls._schema_cache = pq.read_schema(first)
        return [c for c in required_cols if c in cls._schema_cache.names]

    @staticmethod
    def _default_transform():
        '''Resize → ToTensor([0,1]) → map to xrv [-1024, 1024] range.'''
        from torchvision import transforms as T
        return T.Compose([
            T.Resize((cfg.img_size, cfg.img_size)),
            T.ToTensor(),
            T.Lambda(lambda x: x * 2048 - 1024),
        ])

    # ── Lazy image cache: holds at most 2 shards in memory ──

    def _get_image(self, shard_idx, row_idx):
        '''Lazy-load image from parquet with LRU cache.'''
        if shard_idx not in self._shard_cache:
            spath = self.shard_paths[shard_idx]
            # Read ALL columns since we need 'image' but can't know if there's
            # only one shard cache — read only the image column
            df_img = pd.read_parquet(spath, columns=[cfg.col_image])
            self._shard_cache[shard_idx] = df_img
            self._shard_cache_order.append(shard_idx)
            # Evict oldest if > 2 shards cached
            if len(self._shard_cache_order) > 2:
                oldest = self._shard_cache_order.pop(0)
                del self._shard_cache[oldest]
        img_bytes = self._shard_cache[shard_idx].iloc[row_idx][cfg.col_image]
        return self._decode_image(img_bytes)

    @staticmethod
    def _decode_image(img_bytes):
        '''Decode JPEG/PNG bytes to PIL grayscale image.'''
        if isinstance(img_bytes, dict):
            img_bytes = img_bytes.get('bytes') or open(img_bytes['path'], 'rb').read()
        if isinstance(img_bytes, str):
            img = Image.open(img_bytes)
        else:
            img = Image.open(io.BytesIO(bytes(img_bytes)))
        if img.mode != 'L':
            img = img.convert('L')
        return img

    def __len__(self):
        return len(self.patient_groups)

    def __getitem__(self, idx):
        g = self.patient_groups[idx]
        T = min(g['length'], self.max_seq_len)
        L = self.max_seq_len

        imgs = torch.zeros(L, 1, self.img_size, self.img_size, dtype=torch.float32)
        for t in range(T):
            try:
                pil_img = self._get_image(g['shard_idxs'][t], g['row_idxs'][t])
                imgs[t] = self.transform(pil_img)
            except Exception as e:
                pass  # leave as zeros

        labels = torch.full((L, len(self.findings)), -100, dtype=torch.long)
        for t in range(T):
            for f in range(len(self.findings)):
                labels[t, f] = g['labels'][t][f]

        physician_ids = torch.full((L,), -1, dtype=torch.long)
        physician_ids[:T] = torch.tensor(g['physician_ids'][:T])

        ref = g['dates'][0]
        time_gaps = torch.zeros(L, dtype=torch.float32)
        for t in range(T):
            time_gaps[t] = float(max((g['dates'][t] - ref).days, 0))
        absent_mask = torch.zeros(L, dtype=torch.bool)
        absent_mask[T:] = True

        return {
            'images': imgs,
            'labels': labels,
            'physician_ids': physician_ids,
            'time_gaps': time_gaps,
            'absent_mask': absent_mask,
            'length': T,
            'patient_id': g['patient_id'],
        }


In [6]:
# Quick dataset test — load one sample and check image stats
if not sorted(Path(cfg.parquet_dir).glob(cfg.parquet_pattern)):
    print('SKIP dataset test — set cfg.parquet_dir first')
else:
    ds = ParquetCXRDataset(cfg.parquet_dir, cfg.parquet_pattern,
                            cfg.findings, 16, cfg.img_size, fold=0, fraction=0.01)
    sample = ds[0]
    imgs = sample['images']
    print(f'Sample images shape: {imgs.shape}')
    print(f'Image value range: [{imgs.min():.2f}, {imgs.max():.2f}]')
    print(f'Image mean: {imgs.mean():.4f}, std: {imgs.std():.4f}')
    print(f'Labels: {sample["labels"][0]}')
    print(f'Length: {sample["length"]}')
    print(f'Patient: {sample["patient_id"]}')
    del ds, sample


Scanning metadata:   0%|          | 0/59 [00:00<?, ?it/s]

Assembling sequences:   0%|          | 0/96 [00:00<?, ?it/s]

Patients: 58, Visits: min=2 mean=6.2 median=4 max=24
Sample images shape: torch.Size([16, 1, 224, 224])
Image value range: [-1024.00, 1024.00]
Image mean: 198.4684, std: 467.2932
Labels: tensor([0, 2, 2, 0, 2, 2, 2, 2])
Length: 4
Patient: 6536c4c377bb55946b6ad7c308811011056d6bced5bbb24a2b980d79a6381809


In [7]:
class ForecastingEvalDataset(Dataset):
    '''Forecasting split: last visit as target, earlier visits as context.'''
    def __init__(self, parquet_dir, parquet_pattern='*.parquet',
                 findings=None, max_seq_len=64, img_size=224,
                 transform=None, fold=None):
        self.base = ParquetCXRDataset(parquet_dir, parquet_pattern,
                                        findings, max_seq_len, img_size,
                                        transform, fold=fold)
        self._eligible = [g for g in self.base.patient_groups if g['length'] >= 2]

    def __len__(self):
        return len(self._eligible)

    def __getitem__(self, idx):
        g = self._eligible[idx]

        T = g['length']
        L = self.base.max_seq_len
        T_ctx = T - 1

        imgs = torch.zeros(L, 1, self.base.img_size, self.base.img_size, dtype=torch.float32)
        for t in range(T_ctx):
            try:
                pil = self.base._get_image(g['shard_idxs'][t], g['row_idxs'][t])
                imgs[t] = self.base.transform(pil)
            except Exception:
                pass

        labels = torch.full((L, len(self.base.findings)), -100, dtype=torch.long)
        for t in range(T_ctx):
            for f in range(len(self.base.findings)):
                labels[t, f] = g['labels'][t][f]

        target = torch.tensor(g['labels'][T - 1], dtype=torch.long)

        physician_ids = torch.full((L,), -1, dtype=torch.long)
        physician_ids[:T_ctx] = torch.tensor(g['physician_ids'][:T_ctx])

        ref = g['dates'][0]
        time_gaps = torch.zeros(L, dtype=torch.float32)
        for t in range(T_ctx):
            time_gaps[t] = float(max((g['dates'][t] - ref).days, 0))
        time_gaps[T_ctx:] = float(max((g['dates'][T - 1] - ref).days, 0))

        absent_mask = torch.zeros(L, dtype=torch.bool)
        absent_mask[T_ctx:] = True

        return {
            'images': imgs, 'labels': labels, 'target': target,
            'physician_ids': physician_ids, 'time_gaps': time_gaps,
            'absent_mask': absent_mask, 'context_length': T_ctx,
            'patient_id': g['patient_id'],
        }


In [8]:
def longitudinal_collate(batch):
    keys = batch[0].keys()
    out = {}
    for k in keys:
        vals = [b[k] for b in batch]
        if isinstance(vals[0], torch.Tensor):
            out[k] = torch.stack(vals, dim=0)
        else:
            out[k] = vals
    return out


class ParquetDataModule(LightningDataModule):
    def __init__(self, parquet_dir, parquet_pattern='*.parquet',
                 findings=None, max_seq_len=8, img_size=224,
                 batch_size=2, num_workers=0, fold=0, fraction=0.01):
        super().__init__()
        self.parquet_dir = parquet_dir
        self.parquet_pattern = parquet_pattern
        self.findings = findings or []
        self.max_seq_len = max_seq_len
        self.img_size = img_size
        self.batch_size = batch_size
        self.num_workers = 0
        self.fold = fold
        self.fraction = fraction

    def setup(self, stage=None):
        self.train_ds = ParquetCXRDataset(
            self.parquet_dir, self.parquet_pattern,
            self.findings, self.max_seq_len, self.img_size,
            fold=self.fold, fraction=self.fraction,
        )

        val_fold = (self.fold + 1) % cfg.num_folds

        self.val_ds = ParquetCXRDataset(
            self.parquet_dir, self.parquet_pattern,
            self.findings, self.max_seq_len, self.img_size,
            fold=val_fold, fraction=self.fraction,
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=0,
            collate_fn=longitudinal_collate,
            pin_memory=False,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=longitudinal_collate,
            pin_memory=False,
        )

    def test_dataloader(self):
        return self.val_dataloader()

## 4. TorchXRayVision Pretrained Backbone

Uses `densenet121-res224-all` — DenseNet-121 pretrained on 5 CXR datasets
(NIH, CheXpert, MIMIC-CXR, PadChest, RSNA). `model.features2()` → 1024-dim.
Install: `pip install torchxrayvision`

In [9]:
class XRVBackbone(nn.Module):
    def __init__(self, weights='densenet121-res224-all', feat_dim=1024):
        super().__init__()
        if not HAS_XRV:
            raise ImportError('pip install torchxrayvision')
        self.base = xrv.models.DenseNet(weights=weights)
        self.feat_dim = feat_dim
        for p in self.base.parameters():
            p.requires_grad = False

    def unfreeze(self):
        for p in self.base.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.base.features2(x)  # (B, 1024)


class TVBackbone(nn.Module):
    def __init__(self, feat_dim=2048):
        super().__init__()
        from torchvision.models import resnet50, ResNet50_Weights
        base = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.feat_dim = feat_dim
        self.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        with torch.no_grad():
            self.conv1.weight.data = base.conv1.weight.mean(dim=1, keepdim=True)
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.conv1(x); x = self.bn1(x); x = self.relu(x); x = self.maxpool(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.pool(x).flatten(1)


def create_backbone(backbone_type='xrv', **kw):
    if backbone_type == 'xrv':
        return XRVBackbone(kw.get('weights', 'densenet121-res224-all'),
                           kw.get('feat_dim', 1024))
    elif backbone_type == 'resnet50':
        return TVBackbone(kw.get('feat_dim', 2048))
    raise ValueError(f'Unknown backbone: {backbone_type}')


In [10]:
# Eagerly download xrv weights to verify connectivity
print('Downloading xrv backbone...', end=' ', flush=True)
_ = xrv.models.DenseNet(weights='densenet121-res224-all')
print('OK')


If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
OK


## 5. CXR Augmentations

In [11]:
class CXRTransform:
    '''Rotation ±10° + horizontal flip. No color jitter/crop (MoCo-CXR).'''
    def __init__(self, img_size=224, augment=False):
        from torchvision import transforms as T
        ts = [T.Resize((img_size, img_size)), T.Grayscale(1)]
        if augment:
            ts += [T.RandomRotation(10, interpolation=T.InterpolationMode.BILINEAR, fill=0),
                   T.RandomHorizontalFlip(0.5)]
        ts.append(T.ToTensor())
        self.tfm = T.Compose(ts)
    def __call__(self, img):
        return self.tfm(img) * 2048 - 1024


## 6. MoCo (Stage 1, Optional)

Optional if using xrv backbone. Run for domain-adaptation on your data.

In [12]:
class MoCo(nn.Module):
    def __init__(self, backbone, feat_dim=1024, queue_size=65536,
                 momentum=0.999, temperature=0.07, mlp_dim=256):
        super().__init__()
        self.feat_dim = feat_dim
        self.queue_size = queue_size
        self.momentum = momentum
        self.temperature = temperature
        self.encoder_q = nn.Sequential(backbone, nn.Linear(feat_dim, mlp_dim),
                                        nn.ReLU(), nn.Linear(mlp_dim, mlp_dim))
        self.encoder_k = copy.deepcopy(self.encoder_q)
        for p in self.encoder_k.parameters():
            p.requires_grad = False

        self.register_buffer('queue', F.normalize(torch.randn(mlp_dim, queue_size), dim=0))
        self.register_buffer('queue_ptr', torch.zeros(1, dtype=torch.long))
        self.register_buffer('queue_pids', torch.full((queue_size,), -1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update(self):
        for p_q, p_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            p_k.data = p_k.data * self.momentum + p_q.data * (1 - self.momentum)

    def forward(self, imgs_q, imgs_k, patient_ids):
        q = self.encoder_q(imgs_q)
        with torch.no_grad():
            self._momentum_update()
            k = self.encoder_k(imgs_k)

        q, k = F.normalize(q, dim=1), F.normalize(k, dim=1)
        l_pos = (q * k).sum(1, keepdim=True)
        l_neg = torch.mm(q, self.queue.detach())

        mask = patient_ids[:, None] == self.queue_pids[None, :]
        l_neg[mask] = -1e9

        logits = torch.cat([l_pos, l_neg], 1) / self.temperature
        labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)

        eff_neg = (~mask).sum(1).float().mean()

        B = k.shape[0]
        ptr = int(self.queue_ptr)

        if ptr + B <= self.queue_size:
            self.queue[:, ptr:ptr + B] = k.T
            self.queue_pids[ptr:ptr + B] = patient_ids
        else:
            first = self.queue_size - ptr
            self.queue[:, ptr:] = k[:first].T
            self.queue_pids[ptr:] = patient_ids[:first]
            self.queue[:, :B-first] = k[first:].T
            self.queue_pids[:B-first] = patient_ids[first:]

        self.queue_ptr[0] = (ptr + B) % self.queue_size

        return F.cross_entropy(logits, labels), eff_neg


class MoCoLightningModule(LightningModule):
    def __init__(self, backbone_type='xrv', feat_dim=1024, lr=0.03, max_epochs=100):
        super().__init__()
        self.save_hyperparameters()
        self.moco = MoCo(
            create_backbone(backbone_type, feat_dim=feat_dim),
            feat_dim,
            cfg.moco_queue_size,
            cfg.moco_momentum,
            cfg.moco_temperature,
        )
        self.lr, self.max_epochs = lr, max_epochs

    def training_step(self, batch, _):
        imgs = batch['images']
        B = imgs.shape[0]
        vi = batch['absent_mask'].long().argmin(1)
        bi = torch.arange(B, device=imgs.device)

        im_q, im_k = imgs[bi, vi], imgs[bi, vi]
        pids = batch['patient_id']

        if isinstance(pids, list):
            pids = torch.tensor([hash(str(p)) % 1000000 for p in pids], device=imgs.device)

        loss, eff = self.moco(im_q, im_k, pids)

        self.log('loss', loss, prog_bar=True)
        self.log('eff_neg', eff)

        return loss

    def configure_optimizers(self):
        opt = torch.optim.SGD(
            self.moco.encoder_q.parameters(),
            lr=self.lr,
            weight_decay=1e-4,
            momentum=0.9,
        )
        return [opt], [CosineAnnealingLR(opt, self.max_epochs)]

## 7. Temporal Transformer

In [13]:
class Time2Vec(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(1, 1)
        self.periodic = nn.Linear(1, d_model - 1)
    def forward(self, t):
        t = t.unsqueeze(-1).float()
        return torch.cat([self.linear(t), torch.sin(self.periodic(t))], -1)


class TemporalTransformer(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=2,
                 dim_feedforward=512, max_len=64, num_findings=8):
        super().__init__()
        self.time2vec = Time2Vec(d_model)
        self.absent_embed = nn.Parameter(torch.randn(1, 1, d_model))
        self.input_proj = nn.Linear(2 * d_model, d_model)
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.1)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                            0.1, 'gelu', batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers)
        self.finding_projs = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(num_findings)])

    def forward(self, feats, time_gaps, absent_mask, causal=False):
        B, L, D = feats.shape
        te = self.time2vec(time_gaps) + self.pos[:, :L]
        m = absent_mask.unsqueeze(-1).float()
        h = feats * (1 - m) + self.absent_embed * m
        h = self.input_proj(torch.cat([h, te], -1))
        am = None
        if causal:
            am = torch.triu(torch.ones(L, L, dtype=torch.bool, device=h.device), 1)
        h = self.enc(h, mask=am, src_key_padding_mask=absent_mask)
        fh = torch.stack([p(h) for p in self.finding_projs], -2)  # (B, L, 8, D)
        return h, fh


## 8. Causal GAT

In [14]:
def dag_mask(n=8):
    dag = torch.zeros(n, n, dtype=torch.bool)
    for i, j in [(0,1),(1,2),(1,3),(1,4),(1,5)]:
        dag[i, j] = True
    return dag


class CausalGAT(nn.Module):
    def __init__(self, d_model=256, gat_dim=64, n_heads=4, n_nodes=8):
        super().__init__()
        self.dag = dag_mask(n_nodes)
        self.proj = nn.Linear(d_model, gat_dim * n_heads)
        self.n_heads, self.gat_dim, self.n_nodes = n_heads, gat_dim, n_nodes
        self.a = nn.Parameter(torch.randn(n_heads, 2 * gat_dim))
        self.out = nn.Linear(gat_dim * n_heads, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(0.1)

    def forward(self, fh, intervene=None):
        B, L, N, D = fh.shape
        h = self.proj(fh).view(B, L, N, self.n_heads, self.gat_dim)
        adj = self.dag.to(h.device)
        if intervene is not None:
            adj[:, intervene] = False
        outs = []
        for hi in range(self.n_heads):
            hi_ = h[:, :, :, hi].reshape(B * L, N, self.gat_dim)
            a_in = torch.cat([hi_.unsqueeze(2).expand(-1,-1,N,-1),
                              hi_.unsqueeze(1).expand(-1,N,-1,-1)], -1)
            e = F.leaky_relu((a_in @ self.a[hi]).squeeze(-1), 0.2)
            e[~adj.unsqueeze(0).expand_as(e)] = -1e9
            attn = self.drop(F.softmax(e, -1))
            outs.append((attn @ hi_).view(B, L, N, self.gat_dim))
        return self.norm(self.drop(self.out(torch.cat(outs, -1))) + fh)


class MLPBaseline(nn.Module):
    def __init__(self, d_model=256, n_nodes=8):
        super().__init__()
        self.mlps = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(0.1))
            for _ in range(n_nodes)])
    def forward(self, fh):
        return torch.stack([m(fh[:,:,i]) for i, m in enumerate(self.mlps)], 2)


## 9. CORAL Ordinal Head

In [15]:
class CORALHead(nn.Module):
    def __init__(self, d_model=256, K=4, n_findings=8):
        super().__init__()
        self.linear = nn.Linear(d_model, K)
        self.K, self.n_findings = K, n_findings

    def forward(self, fh, bias=None):
        logits = self.linear(fh)
        if bias is not None:
            logits = logits + bias
        thresholds = torch.cumsum(F.softplus(logits), dim=-1)
        probs = torch.sigmoid(thresholds)
        zeros = torch.zeros_like(probs[..., :1])
        ones = torch.ones_like(probs[..., :1])
        padded = torch.cat([zeros, probs, ones], dim=-1)
        dist = padded[..., 1:] - padded[..., :-1]
        return dist.clamp(1e-6) / dist.clamp(1e-6).sum(-1, keepdim=True)


## 10. Physician Bias

In [16]:
class PhysicianBias(nn.Module):
    def __init__(self, n_physicians=500, emb_dim=32, K=4, n_findings=8):
        super().__init__()
        self.emb = nn.Embedding(n_physicians, emb_dim, padding_idx=0)
        self.proj = nn.Linear(emb_dim, K)
    def forward(self, physician_ids):
        ids = physician_ids.clamp(min=0)
        return self.proj(self.emb(ids)).unsqueeze(-2).expand(-1, -1, 8, -1)


## 11. Progression Forecaster (N4)

In [17]:
class ProgressionForecaster(nn.Module):
    def __init__(self, d_model=256, n_findings=8, n_levels=5):
        super().__init__()
        self.label_emb = nn.Embedding(n_levels, d_model)
        self.proj = nn.Linear(d_model * n_findings, d_model)
        layer = nn.TransformerEncoderLayer(d_model, 2, 512, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, 1)
        self.head = nn.Linear(d_model, n_findings * n_levels)
        self.nf, self.nl = n_findings, n_levels

    def forward(self, past_labels, hidden):
        B, T = past_labels.shape[:2]
        le = self.label_emb(past_labels.clamp(min=0)).view(B, T, -1)
        h = hidden + self.proj(le)
        m = torch.triu(torch.ones(T, T, dtype=torch.bool, device=h.device), 1)
        h = self.enc(h, mask=m)
        return F.softmax(self.head(h[:, -1]).view(B, self.nf, self.nl), -1)


## 12. TAIX Model (Full Pipeline)

In [18]:
class TAIXModel(nn.Module):
    def __init__(self, backbone_type='xrv', feat_dim=1024,
                 d_model=256, nhead=4, n_layers=2,
                 use_gat=True, n_findings=8, n_thresholds=4,
                 img_size=224, max_seq_len=64):
        super().__init__()
        self.img_size = img_size
        self.max_seq_len = max_seq_len
        self.backbone = create_backbone(backbone_type, feat_dim=feat_dim)
        self.backbone_proj = nn.Linear(feat_dim, d_model)
        self.temporal = TemporalTransformer(d_model, nhead, n_layers, 512, max_seq_len, n_findings)
        self.graph = CausalGAT(d_model, 64, 4, n_findings) if use_gat else MLPBaseline(d_model, n_findings)
        self.coral = CORALHead(d_model, n_thresholds, n_findings)
        self.phys_bias = PhysicianBias(cfg.num_physicians, cfg.physician_emb_dim, n_thresholds, n_findings)
        self.forecaster = ProgressionForecaster(d_model, n_findings, n_thresholds + 1)
        self.use_gat = use_gat

    def forward(self, images, time_gaps, absent_mask, physician_ids,
                causal=False, intervene=None, ret_hidden=False):
        B, L = images.shape[:2]
        flat = images.view(B * L, 1, self.img_size, self.img_size)
        chunks = flat.split(64)
        feats = torch.cat([self.backbone(c) for c in chunks], dim=0).view(B, L, -1)
        feats = self.backbone_proj(feats)
        h, fh = self.temporal(feats, time_gaps, absent_mask, causal)
        if self.use_gat:
            gf = self.graph(fh, intervene=intervene)
        else:
            gf = self.graph(fh)
        dist = self.coral(gf, bias=self.phys_bias(physician_ids))
        if ret_hidden:
            return dist, h, gf
        return dist

    def intervene(self, images, tg, am, pid, node):
        return (self(images, tg, am, pid),
                self(images, tg, am, pid, intervene=node))

    def forecast(self, images, tg, am, pid, labels):
        dist, h, _ = self(images, tg, am, pid, causal=True, ret_hidden=True)
        return self.forecaster(labels[:, :labels.shape[1]], h), dist


In [19]:
class TAIXLightningModule(LightningModule):
    def __init__(self, backbone_type='xrv', feat_dim=1024,
                 use_gat=True, fcast_weight=0.0, lr=1e-4,
                 wd=1e-5, max_epochs=100, freeze_bn=True):
        super().__init__(); self.save_hyperparameters()
        self.model = TAIXModel(backbone_type, feat_dim, cfg.d_model,
                                cfg.nhead, cfg.transformer_layers,
                                use_gat, len(cfg.findings), cfg.num_thresholds,
                                cfg.img_size, cfg.max_seq_len)
        self.fcast_weight, self.lr, self.wd = fcast_weight, lr, wd
        self.max_epochs = max_epochs
        if freeze_bn:
            for p in self.model.backbone.parameters():
                p.requires_grad = False

    @staticmethod
    def _loss(pred, labels, absent_mask):
        B, L, N, K = pred.shape
        tgt = labels.view(B, L, N)
        m = (tgt != -100) & ~absent_mask.unsqueeze(-1)
        if m.sum() == 0:
            return torch.tensor(0., device=pred.device, requires_grad=True)
        loss = F.nll_loss(pred.clamp(1e-8).log().reshape(-1, K),
                          tgt.reshape(-1).clamp(min=0), reduction='none').reshape(B, L, N)
        loss[~m] = 0.
        return loss.sum() / m.sum()

    @staticmethod
    def _fcast_loss(pred, target):
        B, N, K = pred.shape
        tgt = target.view(B, N)
        m = tgt != -100
        if m.sum() == 0:
            return torch.tensor(0., device=pred.device, requires_grad=True)
        oh = F.one_hot(tgt.clamp(min=0), K).float()
        lp = (pred + 1e-8).log()
        kl = (oh * (oh.clamp(1e-8).log() - lp)).sum(-1)
        kl[~m] = 0.
        return kl.sum() / m.sum()

    def training_step(self, batch, _):
        p = self.model(batch['images'], batch['time_gaps'],
                       batch['absent_mask'], batch['physician_ids'])
        loss = self._loss(p, batch['labels'], batch['absent_mask'])
        if 'target' in batch and self.fcast_weight > 0:
            nxt, _ = self.model.forecast(batch['images'], batch['time_gaps'],
                                          batch['absent_mask'], batch['physician_ids'],
                                          batch['labels'])
            fl = self._fcast_loss(nxt, batch['target'])
            loss = loss + self.fcast_weight * fl
            self.log('fcast', fl)
        self.log('loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        p = self.model(batch['images'], batch['time_gaps'],
                       batch['absent_mask'], batch['physician_ids'])
        loss = self._loss(p, batch['labels'], batch['absent_mask'])
        self.log('val_loss', loss, prog_bar=True)
        pc = p.argmax(-1); m = (batch['labels']!=-100)&~batch['absent_mask'].unsqueeze(-1)
        if m.sum() > 0:
            self.log('val_acc', (pc[m]==batch['labels'][m]).float().mean(), prog_bar=True)
        return loss

    def configure_optimizers(self):
        decay, no_decay = [], []
        for n, p in self.model.named_parameters():
            if not p.requires_grad: continue
            (no_decay if any(k in n for k in ['bias','norm','embed']) else decay).append(p)
        opt = torch.optim.AdamW([{'params': decay, 'weight_decay': self.wd},
                                 {'params': no_decay, 'weight_decay': 0.}], lr=self.lr)
        return {'optimizer': opt, 'lr_scheduler': CosineAnnealingLR(opt, self.max_epochs),
                'gradient_clip_val': 1.0}


## 13. Training Runner

In [20]:
def run_training(cfg, backbone_type='xrv', use_gat=True,
                 freeze_bn=True, fold=0, fraction=0.01,
                 fcast_weight=0.0, seed=42):

    L.seed_everything(seed)

    dm = ParquetDataModule(
        cfg.parquet_dir,
        cfg.parquet_pattern,
        cfg.findings,
        cfg.max_seq_len,
        cfg.img_size,
        cfg.sup_batch_size,
        2,
        fold,
        fraction,
    )
    dm.setup()

    model = TAIXLightningModule(
        backbone_type,
        cfg.feat_dim,
        use_gat,
        fcast_weight,
        cfg.sup_lr,
        cfg.sup_weight_decay,
        cfg.sup_epochs,
        freeze_bn,
    )

    tag = f'{backbone_type}_gat{use_gat}_bn{freeze_bn}'
    if fcast_weight:
        tag += f'_fcast{fcast_weight}'
    if fraction < 1:
        tag += f'_frac{fraction}'

    rn = f'taix_SAFE_{tag}_f{fold}_s{seed}'

    ckpt = ModelCheckpoint(
        dirpath=cfg.output_dir / rn,
        filename='{epoch}-{val_loss:.4f}',
        monitor='val_loss',
        mode='min',
        save_last=True,
        save_top_k=1,
    )

    trainer = L.Trainer(
        max_epochs=cfg.sup_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        strategy='auto',
        sync_batchnorm=False,
        precision='16-mixed' if torch.cuda.is_available() else '32-true',
        gradient_clip_val=1.0,
        callbacks=[ckpt, LearningRateMonitor(logging_interval='epoch')],
        log_every_n_steps=5,
        enable_progress_bar=True,
    )

    trainer.fit(model, dm)

    return model, dm, ckpt

## 14. N2: do-Calculus Intervention

In [21]:
@torch.no_grad()
def evaluate_intervention(model, dl, findings, device='cpu'):
    model = model.to(device).eval()
    deltas = {f: [] for f in findings}
    for batch in dl:
        im, tg, am, pid = [batch[k].to(device) for k in ['images','time_gaps','absent_mask','physician_ids']]
        for node in range(len(findings)):
            fac, cf = model.model.intervene(im, tg, am, pid, node)
            for f in range(len(findings)):
                deltas[findings[f]].append((cf[:,:,f]-fac[:,:,f]).abs().mean().item())
    print('=== do-Calculus ===')
    for f in findings:
        print(f'  {f}: {np.mean(deltas[f]):.4f}±{np.std(deltas[f]):.4f}')
    return deltas


## 15. N4: Forecasting Evaluation

In [22]:
@torch.no_grad()
def evaluate_forecasting(model, dl, findings, device='cpu'):
    model = model.to(device).eval()
    pp, tt, ll = [], [], []
    for batch in dl:
        im, lb, tg, am, pid = [batch[k].to(device) for k in ['images','labels','time_gaps','absent_mask','physician_ids']]
        nxt, _ = model.model.forecast(im, tg, am, pid, lb)
        pp.append(nxt.argmax(-1).cpu()); tt.append(batch['target'].cpu()); ll.extend(batch['context_length'])
    p = torch.cat(pp).numpy(); t = torch.cat(tt).numpy(); l = np.array(ll)
    for fi, fn in enumerate(findings):
        m = t[:, fi] >= 0
        if m.sum() < 2: continue
        rho, _ = scipy.stats.spearmanr(p[m, fi], t[m, fi])
        print(f'  {fn}: ρ={rho:.4f}')
        for sn, (lo, hi) in [('2-3',(2,3)),('4-10',(4,10)),('10+',(10,999))]:
            idx = (l >= lo) & (l <= hi)
            if idx.sum() < 2: continue
            sm = t[idx, fi] >= 0
            if sm.sum() < 2: continue
            sr, _ = scipy.stats.spearmanr(p[idx][sm, fi], t[idx][sm, fi])
            print(f'    [{sn}]: ρ={sr:.4f} (n={sm.sum()})')


## 16. Deep Ensemble Uncertainty

In [23]:
def run_ensemble(cfg, backbone_type='xrv', use_gat=True, fold=0):
    return [run_training(cfg, backbone_type, use_gat, True, fold, seed=s) for s in cfg.ensemble_seeds]

@torch.no_grad()
def ensemble_uncertainty(models, dl, device='cpu'):
    for m in models: m.to(device).eval()
    var = []
    for batch in dl:
        im, tg, am, pid = [batch[k].to(device) for k in ['images','time_gaps','absent_mask','physician_ids']]
        var.append(np.stack([m.model(im, tg, am, pid).cpu().numpy() for m in models]).var(0).mean(-1))
    v = np.concatenate(var); print(f'Mean uncertainty: {v.mean():.6f}'); return v


## 17. Orchestration

In [24]:
if __name__ == '__main__':
    cfg.parquet_dir = Path('/kaggle/input/datasets/ahmedalkasaby/taix-ray-dataset')
    cfg.output_dir = Path('/kaggle/working/')

    cfg.sup_epochs = 3
    cfg.max_seq_len = 8
    cfg.sup_batch_size = 4

    ds = ParquetCXRDataset(
        cfg.parquet_dir,
        findings=cfg.findings,
        max_seq_len=cfg.max_seq_len,
        img_size=cfg.img_size,
        fraction=0.01,
    )

    print("Dataset size:", len(ds))
    print("Sample keys:", ds[0].keys())

    model, dm, ckpt = run_training(
        cfg,
        'xrv',
        use_gat=True,
        fraction=0.01,
        seed=42,
    )

    print("Training finished")
    print("Best checkpoint:", ckpt.best_model_path)
    print("Last checkpoint:", ckpt.last_model_path)

Scanning metadata:   0%|          | 0/59 [00:00<?, ?it/s]

Assembling sequences:   0%|          | 0/477 [00:00<?, ?it/s]

Patients: 319, Visits: min=2 mean=6.6 median=4 max=65
Dataset size: 319


Seed set to 42


Sample keys: dict_keys(['images', 'labels', 'physician_ids', 'time_gaps', 'absent_mask', 'length', 'patient_id'])


...

┏━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ TAIXModel │ 10.2 M │ train │     0 │
└───┴───────┴───────────┴────────┴───────┴───────┘

Trainable params: 3.2 M                                                                                            
Non-trainable params: 7.0 M                                                                                        
Total params: 10.2 M                                                                                               
Total estimated model params size (MB): 40.624                                                                     
Modules in train mode: 65                                                                                          
Modules in eval mode: 433                                                                                          
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=3` reached.


Training finished
Best checkpoint: /kaggle/working/taix_SAFE_xrv_gatTrue_bnTrue_frac0.01_f0_s42/epoch=0-val_loss=nan.ckpt
Last checkpoint: /kaggle/working/taix_SAFE_xrv_gatTrue_bnTrue_frac0.01_f0_s42/last.ckpt


In [25]:
severity_names = {
    0: "normal/absent",
    1: "minimal",
    2: "mild",
    3: "moderate",
    4: "severe",
}

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

batch = next(iter(dm.val_dataloader()))

images = batch["images"].to(device)
time_gaps = batch["time_gaps"].to(device)
absent_mask = batch["absent_mask"].to(device)
physician_ids = batch["physician_ids"].to(device)

with torch.no_grad():
    out = model.model(
        images,
        time_gaps,
        absent_mask,
        physician_ids
    )

print("Output type:", type(out))

if isinstance(out, dict):
    print("Output keys:", out.keys())

    if "probs" in out:
        preds = out["probs"].argmax(dim=-1).cpu()
    elif "logits" in out:
        preds = out["logits"].argmax(dim=-1).cpu()
    else:
        raise KeyError(f"Could not find probs/logits. Available keys: {out.keys()}")
else:
    preds = out.argmax(dim=-1).cpu()

print("Pred shape:", preds.shape)

for b in range(preds.shape[0]):
    print(f"\nPatient {b} | ID: {batch['patient_id'][b]}")

    for t in range(preds.shape[1]):
        if batch["absent_mask"][b, t].item():
            continue

        print(f"  Visit {t + 1}")

        for i, finding in enumerate(cfg.findings):
            sev = preds[b, t, i].item()
            print(f"    {finding}: {sev} = {severity_names.get(sev, 'unknown')}")

Output type: <class 'torch.Tensor'>
Pred shape: torch.Size([4, 8, 8])

Patient 0 | ID: 7e48451e2259a8962696940234fffac7323de4cee5c243f7836fbf6b5538bde3
  Visit 1
    HeartSize: 0 = normal/absent
    PulmonaryCongestion: 0 = normal/absent
    PleuralEffusion_Right: 0 = normal/absent
    PleuralEffusion_Left: 0 = normal/absent
    PulmonaryOpacities_Right: 0 = normal/absent
    PulmonaryOpacities_Left: 0 = normal/absent
    Atelectasis_Right: 0 = normal/absent
    Atelectasis_Left: 0 = normal/absent
  Visit 2
    HeartSize: 0 = normal/absent
    PulmonaryCongestion: 0 = normal/absent
    PleuralEffusion_Right: 0 = normal/absent
    PleuralEffusion_Left: 0 = normal/absent
    PulmonaryOpacities_Right: 0 = normal/absent
    PulmonaryOpacities_Left: 0 = normal/absent
    Atelectasis_Right: 0 = normal/absent
    Atelectasis_Left: 0 = normal/absent
  Visit 3
    HeartSize: 0 = normal/absent
    PulmonaryCongestion: 0 = normal/absent
    PleuralEffusion_Right: 0 = normal/absent
    PleuralEffu

In [26]:
batch = next(iter(dm.val_dataloader()))

model.eval()

with torch.no_grad():
    out = model.model(
        batch["images"].to(device),
        batch["time_gaps"].to(device),
        batch["absent_mask"].to(device),
        batch["physician_ids"].to(device),
    )

print(type(out))

if isinstance(out, dict):
    print(out.keys())

    for k, v in out.items():
        if torch.is_tensor(v):
            print(
                k,
                v.shape,
                "nan:", torch.isnan(v).any().item(),
                "min:", float(v.min()),
                "max:", float(v.max()),
            )

<class 'torch.Tensor'>


In [27]:

batch = next(iter(dm.train_dataloader()))

for i, finding in enumerate(cfg.findings):
    vals = batch["labels"][:, :, i]
    vals = vals[vals >= 0]

    print(
        finding,
        sorted(torch.unique(vals).tolist())
    )

HeartSize [0, 1, 2]
PulmonaryCongestion [0, 1, 2]
PleuralEffusion_Right [0, 1, 2, 3]
PleuralEffusion_Left [0, 1, 2]
PulmonaryOpacities_Right [0, 1, 2]
PulmonaryOpacities_Left [0]
Atelectasis_Right [0, 2]
Atelectasis_Left [0, 2]


In [28]:
batch = next(iter(dm.val_dataloader()))

model.eval()

with torch.no_grad():
    out = model.model(
        batch["images"].to(device),
        batch["time_gaps"].to(device),
        batch["absent_mask"].to(device),
        batch["physician_ids"].to(device),
    )

print("Shape:", out.shape)
print("Contains NaN:", torch.isnan(out).any().item())
print("Min:", float(out.min()))
print("Max:", float(out.max()))
print("Mean:", float(out.mean()))

preds = out.argmax(dim=-1).cpu()
print("Unique prediction classes:", torch.unique(preds))

Shape: torch.Size([4, 8, 8, 5])
Contains NaN: False
Min: 0.0024613142013549805
Max: 0.8939195275306702
Mean: 0.20000000298023224
Unique prediction classes: tensor([0])


In [29]:
# =========================
# SECOND EXPERIMENT
# 5% data, 5 epochs, safe Kaggle settings
# =========================

cfg.sup_epochs = 5
cfg.max_seq_len = 8
cfg.sup_batch_size = 2

model, dm, ckpt = run_training(
    cfg,
    'xrv',
    use_gat=True,
    fraction=0.05,
    seed=42
)

print("Training finished")
print("Best checkpoint:", ckpt.best_model_path)
print("Last checkpoint:", ckpt.last_model_path)


# =========================
# CHECK MODEL OUTPUT
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

batch = next(iter(dm.val_dataloader()))

with torch.no_grad():
    out = model.model(
        batch["images"].to(device),
        batch["time_gaps"].to(device),
        batch["absent_mask"].to(device),
        batch["physician_ids"].to(device),
    )

print("\nOutput diagnostics")
print("Shape:", out.shape)
print("Contains NaN:", torch.isnan(out).any().item())
print("Min:", float(out.min()))
print("Max:", float(out.max()))
print("Mean:", float(out.mean()))

preds = out.argmax(dim=-1).cpu()
print("Unique prediction classes:", torch.unique(preds))


# =========================
# PRINT SEVERITY PREDICTIONS
# =========================

severity_names = {
    0: "normal/absent",
    1: "minimal",
    2: "mild",
    3: "moderate",
    4: "severe",
}

for b in range(preds.shape[0]):
    print(f"\nPatient {b} | ID: {batch['patient_id'][b]}")

    for t in range(preds.shape[1]):
        if batch["absent_mask"][b, t].item():
            continue

        print(f"  Visit {t + 1}")

        for i, finding in enumerate(cfg.findings):
            sev = preds[b, t, i].item()
            print(f"    {finding}: {sev} = {severity_names.get(sev, 'unknown')}")

Seed set to 42


Scanning metadata:   0%|          | 0/59 [00:00<?, ?it/s]

Assembling sequences:   0%|          | 0/481 [00:00<?, ?it/s]

Patients: 307, Visits: min=2 mean=6.4 median=4 max=43


Scanning metadata:   0%|          | 0/59 [00:00<?, ?it/s]

...

┏━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ TAIXModel │ 10.2 M │ train │     0 │
└───┴───────┴───────────┴────────┴───────┴───────┘

Trainable params: 3.2 M                                                                                            
Non-trainable params: 7.0 M                                                                                        
Total params: 10.2 M                                                                                               
Total estimated model params size (MB): 40.624                                                                     
Modules in train mode: 65                                                                                          
Modules in eval mode: 433                                                                                          
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=5` reached.


Training finished
Best checkpoint: /kaggle/working/taix_SAFE_xrv_gatTrue_bnTrue_frac0.05_f0_s42/epoch=0-val_loss=nan.ckpt
Last checkpoint: /kaggle/working/taix_SAFE_xrv_gatTrue_bnTrue_frac0.05_f0_s42/last.ckpt

Output diagnostics
Shape: torch.Size([2, 8, 8, 5])
Contains NaN: False
Min: 0.006048083305358887
Max: 0.8823465704917908
Mean: 0.20000000298023224
Unique prediction classes: tensor([0])

Patient 0 | ID: 7e48451e2259a8962696940234fffac7323de4cee5c243f7836fbf6b5538bde3
  Visit 1
    HeartSize: 0 = normal/absent
    PulmonaryCongestion: 0 = normal/absent
    PleuralEffusion_Right: 0 = normal/absent
    PleuralEffusion_Left: 0 = normal/absent
    PulmonaryOpacities_Right: 0 = normal/absent
    PulmonaryOpacities_Left: 0 = normal/absent
    Atelectasis_Right: 0 = normal/absent
    Atelectasis_Left: 0 = normal/absent
  Visit 2
    HeartSize: 0 = normal/absent
    PulmonaryCongestion: 0 = normal/absent
    PleuralEffusion_Right: 0 = normal/absent
    PleuralEffusion_Left: 0 = normal/ab